In [1]:
import json
import logging

import joblib
import pandas as pd

from torch.utils.data import DataLoader, Dataset
from sentence_transformers import (
    InputExample,
    LoggingHandler,
    SentenceTransformer,
    losses,
)

from stemmer import Stemmer, tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

/home/fahmi/freelance/project-2023-amsearch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

In [3]:
## TRAINING PARAMS
STEMMING_AT_TRAIN = False

## BoW & TF-IDF Model

In [4]:
class AMSTokenizer:
    def __init__(self, stm: Stemmer):
        self.stemmer = stm

    def __call__(self, doc):
        return [self.stemmer.stem_ams(word) for word in tokenize(doc)]

### Data Prep

In [5]:
stemmer = Stemmer("../data/sundabaru1-vocab.txt")
amstokenizer = AMSTokenizer(stemmer)

In [6]:
df_corpus = pd.read_json("../data/triplet/triplet.jsonl", lines=True)
df_corpus.head()

,query,positive,negative
0,Kumaha cara ngahontal kahayang dina kahirupan?,"Kahiji, urang kedah sabar sareng henteu janten...",Abdi ngadangu seueur warta ngeunaan jalma anu ...
1,Naon anu kedah dilakukeun lamun gering?,"Lamun gering, ulah rungsing sabab pikiran posi...",Masyarakat ayeuna seueur nganggur di kota nu g...
2,Kumaha cara nyieun amal?,"Ngawitan amal ti hal-hal leutik, sapertos ngab...",Jalma sering nyarita ngeunaan kaékonomian anu ...
3,Kumaha sangkan ngeterkeun diri ka Gusti?,"Mertahankeun ati, pariksa tindakan sorangan, s...","Saurang guru ngajarkeun pentingna ilmu, tapi k..."
4,Naon hartina jadi jalma leutik?,Jadi jalma leutik hartina ulah sombong sareng ...,"Dina pagelaran, anu katinggali gaduh prestasi ..."


In [7]:
df_corpus = pd.read_json("../data/triplet/triplet.jsonl", lines=True)
df_corpus.head()

,query,positive,negative
0,Kumaha cara ngahontal kahayang dina kahirupan?,"Kahiji, urang kedah sabar sareng henteu janten...",Abdi ngadangu seueur warta ngeunaan jalma anu ...
1,Naon anu kedah dilakukeun lamun gering?,"Lamun gering, ulah rungsing sabab pikiran posi...",Masyarakat ayeuna seueur nganggur di kota nu g...
2,Kumaha cara nyieun amal?,"Ngawitan amal ti hal-hal leutik, sapertos ngab...",Jalma sering nyarita ngeunaan kaékonomian anu ...
3,Kumaha sangkan ngeterkeun diri ka Gusti?,"Mertahankeun ati, pariksa tindakan sorangan, s...","Saurang guru ngajarkeun pentingna ilmu, tapi k..."
4,Naon hartina jadi jalma leutik?,Jadi jalma leutik hartina ulah sombong sareng ...,"Dina pagelaran, anu katinggali gaduh prestasi ..."


In [8]:
train_corpus = df_corpus.values.ravel().tolist()
len(train_corpus), train_corpus[0]

(22476, 'Kumaha cara ngahontal kahayang dina kahirupan?')

### Training

In [9]:
MODEL_NAME = "ams-bag_of_words"

if "bag" in MODEL_NAME:
    vsm_model = CountVectorizer(tokenizer=amstokenizer if STEMMING_AT_TRAIN else None)
else:
    vsm_model = TfidfVectorizer(tokenizer=amstokenizer if STEMMING_AT_TRAIN else None)

vsm_model.fit(train_corpus)

CountVectorizer()

In [10]:
joblib.dump(vsm_model, f"../tmp/{MODEL_NAME}.joblib")

['../tmp/ams-bag_of_words.joblib']

## Dense Model

### Data Prep

In [11]:
def stem_sentence(stemmer: Stemmer, text: str) -> str:
    return " ".join([stemmer.stem_ams(t) for t in tokenize(text)])

In [12]:
class TripletDataset(Dataset):
    def __init__(self, dataset_path: str, stemmer: Stemmer = None):
        self.stemmer = stemmer
        with open(dataset_path, "r") as f:
            self.dataset = [json.loads(x) for x in f]

    def __getitem__(self, idx):
        item = self.dataset[idx]

        if self.stemmer:
            return InputExample(
                texts=[
                    stem_sentence(self.stemmer, item["query"]),
                    stem_sentence(self.stemmer, item["positive"]),
                    stem_sentence(self.stemmer, item["negative"]),
                ]
            )

        return InputExample(texts=[item["query"], item["positive"], item["negative"]])

    def __len__(self):
        return len(self.dataset)

In [13]:
train_dataset = TripletDataset("../data/triplet/triplet.jsonl")
# train_dataset = TripletDataset("../data/triplet/triplet.jsonl", stemmer=stemmer)

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)

### Fine-Tuning

In [14]:
model = SentenceTransformer("sentence-transformers/msmarco-distilbert-cos-v5")
model.max_seq_length = 512

2025-04-25 14:46:10 - Use pytorch device_name: cuda
2025-04-25 14:46:10 - Load pretrained SentenceTransformer: sentence-transformers/msmarco-distilbert-cos-v5


In [15]:
train_loss = losses.MultipleNegativesRankingLoss(model=model)

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    use_amp=True,
    epochs=10,
    warmup_steps=10000,
    optimizer_params={"lr": 2e-5},
)

Step,Training Loss
50,2.769800
100,2.801800


In [17]:
model.save(f"../tmp/{MODEL_NAME}")

2025-04-25 14:47:00 - Save model to ../tmp/ams-bag_of_words
